<a href="https://www.kaggle.com/code/moizkalid/ai-based-heart-disease-detection-system-mri?scriptVersionId=314187708" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pip install numpy pandas matplotlib nibabel scikit-learn tensorflow opencv-python-headless

In [2]:
from google.colab import drive
drive.mount('/content/drive')

NotImplementedError: Mounting drive is unsupported in this environment. Use PyDrive2 instead. See examples at https://colab.research.google.com/notebooks/io.ipynb#scrollTo=7taylj9wpsA2.

In [ ]:
import os

dataset_path = '/content/drive/MyDrive/Dataset'
print(os.listdir(dataset_path))

In [ ]:
import os

# Go one level deeper
dataset_path = '/content/drive/MyDrive/Dataset/ACDC_preprocessed'
print(os.listdir(dataset_path))

In [ ]:
# Check number of files in each folder
import os

base_path = '/content/drive/MyDrive/Dataset/ACDC_preprocessed'

for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)
    files = os.listdir(folder_path)
    print(f"{folder}: {len(files)} files")

In [ ]:
slices_path = '/content/drive/MyDrive/Dataset/ACDC_preprocessed/ACDC_training_slices'

# Show first 10 files
files = os.listdir(slices_path)
print(files[:10])

## Step 1: Dataset Preparation ✅

### Dataset: ACDC (Automated Cardiac Diagnosis Challenge)
- Total Images: 1912 MRI slices
- Format: HDF5 (.h5)
- Classes: 5 (NOR, DCM, MINF, RV, HCM)
- Source: Kaggle (archive 3)

### Folder Structure:
- ACDC_training_slices → 1912 files
- ACDC_training_volumes → 200 files
- ACDC_testing_volumes → 100 files

In [ ]:
import h5py

# Open one sample file
sample_file = os.path.join(slices_path, files[0])

with h5py.File(sample_file, 'r') as f:
    print("Keys inside file:", list(f.keys()))
    for key in f.keys():
        print(f"{key} → shape: {f[key].shape}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

with h5py.File(sample_file, 'r') as f:
    image = f['image'][:]
    label = f['label'][:]

# Plot image and label side by side
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(image, cmap='gray')
axes[0].set_title('MRI Image')
axes[0].axis('off')

axes[1].imshow(label, cmap='jet')
axes[1].set_title('Label Mask')
axes[1].axis('off')

plt.suptitle(f'Sample: {files[0]}', fontsize=12)
plt.tight_layout()
plt.show()

## Step 2: Data Preprocessing ✅

### What we did:
- Loaded all 1912 MRI images
- Resized all images to (256, 256)
- Added channel dimension → (1912, 256, 256, 1)
- Created disease labels (5 classes)
- Applied Data Augmentation
  - Rotation: 15 degrees
  - Width/Height shift: 10%
  - Horizontal flip
  - Zoom: 10%
- Saved processed data to Google Drive

### Class Distribution:
- NOR  (Normal)                → 400 images
- DCM  (Dilated Cardiomyopathy)→ 378 images
- MINF (Myocardial Infarction) → 346 images
- RV   (Right Ventricle)       → 340 images
- HCM  (Hypertrophic)          → 448 images

In [ ]:
import h5py
import numpy as np
import os
import cv2

slices_path = '/content/drive/MyDrive/Dataset/ACDC_preprocessed/ACDC_training_slices'

images = []
labels = []

# Standard size for all images
TARGET_SIZE = (256, 256)

for file in os.listdir(slices_path):
    file_path = os.path.join(slices_path, file)
    with h5py.File(file_path, 'r') as f:
        image = f['image'][:]
        label = f['label'][:]

        # Resize to same size
        image = cv2.resize(image, TARGET_SIZE)
        label = cv2.resize(label, TARGET_SIZE,
                          interpolation=cv2.INTER_NEAREST)

        # Normalize image to range [0, 1]
        image = (image - np.min(image)) / (np.max(image) - np.min(image))

        images.append(image)
        labels.append(label)

images = np.array(images)
labels = np.array(labels)

print(f"Total images loaded: {images.shape}")
print(f"Total labels loaded: {labels.shape}")

In [ ]:
# CNN expects shape (samples, height, width, channels)
images = images[..., np.newaxis]
print(f"Images shape after adding channel: {images.shape}")

In [ ]:
print("Unique label values:", np.unique(labels))
print("Label value counts:")
unique, counts = np.unique(labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Class {u}: {c} pixels")

In [ ]:
import re

slices_path = '/content/drive/MyDrive/Dataset/ACDC_preprocessed/ACDC_training_slices'
volumes_path = '/content/drive/MyDrive/Dataset/ACDC_preprocessed/ACDC_training_volumes'

# Check what's inside a volume file
vol_files = os.listdir(volumes_path)
print("Sample volume files:", vol_files[:5])

In [ ]:
# Open one volume file and check its keys
vol_sample = os.path.join(volumes_path, vol_files[0])

with h5py.File(vol_sample, 'r') as f:
    print("Keys inside volume file:", list(f.keys()))
    for key in f.keys():
        data = f[key]
        print(f"{key} → shape: {data.shape}, dtype: {data.dtype}")

In [ ]:
test_path = '/content/drive/MyDrive/Dataset/ACDC_preprocessed/ACDC_testing_volumes'

test_files = os.listdir(test_path)
print("Sample testing files:", test_files[:5])

# Open one and check keys
test_sample = os.path.join(test_path, test_files[0])

with h5py.File(test_sample, 'r') as f:
    print("\nKeys inside testing file:", list(f.keys()))
    for key in f.keys():
        data = f[key]
        print(f"{key} → shape: {data.shape}, dtype: {data.dtype}")

In [ ]:
def get_patient_label(filename):
    # Extract patient number from filename
    patient_num = int(re.search(r'patient(\d+)', filename).group(1))

    if 1 <= patient_num <= 20:
        return 0  # NOR
    elif 21 <= patient_num <= 40:
        return 1  # DCM
    elif 41 <= patient_num <= 60:
        return 2  # MINF
    elif 61 <= patient_num <= 80:
        return 3  # RV
    elif 81 <= patient_num <= 100:
        return 4  # HCM

# Test it
for f in os.listdir(slices_path)[:5]:
    label = get_patient_label(f)
    print(f"{f} → Class: {label}")

In [ ]:
disease_labels = []

for file in os.listdir(slices_path):
    label = get_patient_label(file)
    disease_labels.append(label)

disease_labels = np.array(disease_labels)

print(f"Total labels: {len(disease_labels)}")
print(f"Class distribution:")
unique, counts = np.unique(disease_labels, return_counts=True)
class_names = ['NOR', 'DCM', 'MINF', 'RV', 'HCM']
for u, c in zip(unique, counts):
    print(f"  {class_names[u]}: {c} images")

In [ ]:
import matplotlib.pyplot as plt

class_names = ['NOR', 'DCM', 'MINF', 'RV', 'HCM']
counts = [400, 378, 346, 340, 448]
colors = ['#4CAF50','#2196F3','#FF9800','#E91E63','#9C27B0']

plt.figure(figsize=(8, 5))
bars = plt.bar(class_names, counts, color=colors)
plt.title('Class Distribution', fontsize=14)
plt.xlabel('Heart Condition')
plt.ylabel('Number of Images')

for bar, val in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 5, str(val),
             ha='center', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define augmentation techniques
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)

# Test augmentation on one image
sample_image = images[0:1]  # Take first image

# Generate 5 augmented versions
fig, axes = plt.subplots(1, 6, figsize=(18, 3))

# Show original
axes[0].imshow(images[0, :, :, 0], cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

# Show 5 augmented
for i, batch in enumerate(datagen.flow(sample_image, batch_size=1)):
    axes[i+1].imshow(batch[0, :, :, 0], cmap='gray')
    axes[i+1].set_title(f'Aug {i+1}')
    axes[i+1].axis('off')
    if i == 4:
        break

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# Save images and labels to Google Drive
save_path = '/content/drive/MyDrive/pulse-ai/ml_model/data/processed/'
os.makedirs(save_path, exist_ok=True)

np.save(save_path + 'images.npy', images)
np.save(save_path + 'labels.npy', disease_labels)

print(f"✅ Images saved! Shape: {images.shape}")
print(f"✅ Labels saved! Shape: {disease_labels.shape}")

## Step 3 & 4: Model Building & Training ✅

### Model Architecture:
- 4 Convolutional Blocks
- Batch Normalization after each block
- 2 Fully Connected layers
- Dropout for regularization
- Output: 5 classes (NOR, DCM, MINF, RV, HCM)
- Total Parameters: 13.2 Million

### Training Results:
- Epochs: 21 (Early stopping)
- Best Validation Accuracy: 66.84%
- Best performing class: HCM (98% precision)
- Saved: final_model.keras ✅

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    images, disease_labels,
    test_size=0.2,
    random_state=42,
    stratify=disease_labels
)

# Convert labels to categorical
y_train_cat = to_categorical(y_train, num_classes=5)
y_test_cat = to_categorical(y_test, num_classes=5)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train_cat.shape}")
print(f"y_test:  {y_test_cat.shape}")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

model = Sequential([
    # Block 1
    Conv2D(32, (3,3), activation='relu', input_shape=(256, 256, 1)),
    BatchNormalization(),
    MaxPooling2D(2,2),

    # Block 2
    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    # Block 3
    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    # Block 4
    Conv2D(256, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    # Fully Connected Layers
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dropout(0.3),

    # Output Layer (5 classes)
    Dense(5, activation='softmax')
])

model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Model compiled successfully!")

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Save best model automatically
checkpoint = ModelCheckpoint(
    '/content/drive/MyDrive/pulse-ai/ml_model/models/best_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

# Stop training if no improvement
early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    verbose=1
)

# Train the model
history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_test, y_test_cat),
    epochs=30,
    batch_size=32,
    callbacks=[checkpoint, early_stopping]
)

In [ ]:
import matplotlib.pyplot as plt

# Plot accuracy
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Get predictions
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test_cat, axis=1)

# Classification report
class_names = ['NOR', 'DCM', 'MINF', 'RV', 'HCM']
print(classification_report(y_test_classes, y_pred_classes,
                            target_names=class_names))

# Confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test_classes, y_pred_classes)
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=class_names,
            yticklabels=class_names,
            cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# Save classification report to Drive
report = classification_report(y_test_classes, y_pred_classes,
                               target_names=class_names)

save_path = '/content/drive/MyDrive/pulse-ai/ml_model/'
with open(save_path + 'evaluation_report.txt', 'w') as f:
    f.write(report)

print("✅ Evaluation report saved!")

# Save final model in keras format
model.save(save_path + 'models/final_model.keras')
print("✅ Final model saved!")

## Step 5: Transfer Learning with VGG16 ✅

### Why Transfer Learning?

- Transfer Learning uses pretrained weights
- Trained on millions of images (ImageNet)
- Fine tuned on our MRI data
- Expected accuracy improvement → 80%+

### Model: VGG16
- Pretrained on ImageNet dataset
- Freeze base layers
- Add custom classification head
- Output: 5 classes (NOR, DCM, MINF, RV, HCM)

### Why VGG16?
- Works great for medical images
- Lightweight enough for Colab
- Well tested in research papers

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Flatten, GlobalAveragePooling2D, Input
import tensorflow as tf

# VGG16 expects 3 channels so we convert 1 channel to 3
inputs = Input(shape=(256, 256, 1))
x = tf.keras.layers.Conv2D(3, (1,1), padding='same')(inputs)

# Load pretrained VGG16
base_model = VGG16(weights='imagenet',
                   include_top=False,
                   input_shape=(256, 256, 3))
base_model.trainable = False

x = base_model(x)
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(5, activation='softmax')(x)

vgg_model = Model(inputs, outputs)
vgg_model.summary()

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

vgg_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
checkpoint = ModelCheckpoint(
    '/content/drive/MyDrive/pulse-ai/ml_model/models/best_vgg_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    verbose=1
)

# Train
history_vgg = vgg_model.fit(
    X_train, y_train_cat,
    validation_data=(X_test, y_test_cat),
    epochs=30,
    batch_size=32,
    callbacks=[checkpoint, early_stopping]
)

In [ ]:
# Get predictions
y_pred_vgg = vgg_model.predict(X_test)
y_pred_classes_vgg = np.argmax(y_pred_vgg, axis=1)
y_test_classes = np.argmax(y_test_cat, axis=1)

# Classification report
class_names = ['NOR', 'DCM', 'MINF', 'RV', 'HCM']
print(classification_report(y_test_classes, y_pred_classes_vgg,
                            target_names=class_names))

# Confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test_classes, y_pred_classes_vgg)
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=class_names,
            yticklabels=class_names,
            cmap='Blues')
plt.title('VGG16 Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## Step 6: Model Evaluation ✅

### CNN vs VGG16 Comparison:
- CNN Accuracy  → 67%
- VGG16 Accuracy → 81% ✅

### VGG16 Per Class Results:
- NOR  → 83% precision
- DCM  → 74% precision
- MINF → 72% precision
- RV   → 86% precision
- HCM  → 90% precision

### Saved Files:
- final_vgg_model.keras ✅
- vgg_evaluation_report.txt ✅

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_vgg.history['accuracy'], label='Train')
axes[0].plot(history_vgg.history['val_accuracy'], label='Validation')
axes[0].set_title('VGG16 Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history_vgg.history['loss'], label='Train')
axes[1].plot(history_vgg.history['val_loss'], label='Validation')
axes[1].set_title('VGG16 Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Save final VGG16 model
save_path = '/content/drive/MyDrive/pulse-ai/ml_model/models/'

vgg_model.save(save_path + 'final_vgg_model.keras')
print("✅ Final VGG16 model saved!")

# Save evaluation report
report = classification_report(y_test_classes, y_pred_classes_vgg,
                               target_names=class_names)

with open('/content/drive/MyDrive/pulse-ai/ml_model/vgg_evaluation_report.txt', 'w') as f:
    f.write(report)

print("✅ Evaluation report saved!")

## Step 7: Model Improvement - Fine-Tuning VGG16

In this step, we unfreeze the last block of VGG16 to allow the model to learn high-level features specific to our MRI dataset.

In [ ]:
from tensorflow.keras.optimizers import Adam

# Unfreeze the base model
base_model.trainable = True

# Fine-tune from this layer onwards
# VGG16 has 19 layers. We unfreeze the last convolutional block (block5)
fine_tune_at = 15

# Freeze all the layers before the `fine_tune_at` layer
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Recompile with a very low learning rate
vgg_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

vgg_model.summary()

In [ ]:
# Continue training with fine-tuning
fine_tune_epochs = 20
total_epochs = 30 + fine_tune_epochs

checkpoint_ft = ModelCheckpoint(
    '/content/drive/MyDrive/pulse-ai/ml_model/models/best_vgg_finetuned.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

history_fine = vgg_model.fit(
    X_train, y_train_cat,
    validation_data=(X_test, y_test_cat),
    epochs=total_epochs,
    initial_epoch=history_vgg.epoch[-1],
    batch_size=32,
    callbacks=[checkpoint_ft, early_stopping]
)

## Step 8: Fine-Tuning Evaluation and Comparison

We will now evaluate the fine-tuned model's performance and compare the learning curves.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np

# Get predictions from the fine-tuned model
y_pred_ft = vgg_model.predict(X_test)
y_pred_classes_ft = np.argmax(y_pred_ft, axis=1)
y_test_classes = np.argmax(y_test_cat, axis=1)

# Classification report
print("Fine-tuned VGG16 Classification Report:")
print(classification_report(y_test_classes, y_pred_classes_ft, target_names=class_names))

# Plotting combined history
acc = history_vgg.history['accuracy'] + history_fine.history['accuracy']
val_acc = history_vgg.history['val_accuracy'] + history_fine.history['val_accuracy']
loss = history_vgg.history['loss'] + history_fine.history['loss']
val_loss = history_vgg.history['val_loss'] + history_fine.history['val_loss']

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.plot([len(history_vgg.history['accuracy'])-1, len(history_vgg.history['accuracy'])-1],
         plt.ylim(), label='Start Fine Tuning', linestyle='--')
plt.title('Accuracy Improvement')
plt.legend(loc='lower right')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.plot([len(history_vgg.history['loss'])-1, len(history_vgg.history['loss'])-1],
         plt.ylim(), label='Start Fine Tuning', linestyle='--')
plt.title('Loss Improvement')
plt.legend(loc='upper right')

plt.tight_layout()
plt.show()


# Testing

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from google.colab import files

# Define descriptions to fix NameError
class_descriptions = {
    'NOR': 'Normal: No significant cardiac abnormalities detected.',
    'DCM': 'Dilated Cardiomyopathy: Enlarged and weakened left ventricle.',
    'MINF': 'Myocardial Infarction: Evidence of past heart attack/muscle damage.',
    'RV': 'Right Ventricular Abnormality: Issues affecting the right side of the heart.',
    'HCM': 'Hypertrophic Cardiomyopathy: Thickened heart muscle making it harder to pump.'
}

# Upload image from PC
print("Please upload an MRI image...")
uploaded = files.upload()

# Get uploaded filename
if uploaded:
    image_name = list(uploaded.keys())[0]
    print(f"✅ Uploaded: {image_name}")

    # Preprocess
    img = cv2.imdecode(np.frombuffer(uploaded[image_name], np.uint8),
                       cv2.IMREAD_GRAYSCALE)
    img_resized = cv2.resize(img, (256, 256))
    img_normalized = img_resized / 255.0
    img_input = img_normalized[..., np.newaxis]
    img_input = np.expand_dims(img_input, axis=0)

    # Predict
    prediction = vgg_model.predict(img_input)
    predicted_class = np.argmax(prediction)
    confidence = prediction[0][predicted_class] * 100
    class_name = class_names[predicted_class]

    # Display results
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].imshow(img_resized, cmap='gray')
    axes[0].set_title('Your MRI Image', fontsize=12)
    axes[0].axis('off')

    colors = ['green' if i == predicted_class else 'steelblue'
              for i in range(5)]
    axes[1].barh(class_names, prediction[0] * 100, color=colors)
    axes[1].set_xlabel('Confidence %')
    axes[1].set_title('Prediction Results', fontsize=12)
    axes[1].set_xlim(0, 100)

    for i, v in enumerate(prediction[0] * 100):
        axes[1].text(v + 1, i, f'{v:.1f}%', va='center')

    plt.suptitle(
        f'Predicted: {class_name} ({confidence:.1f}%)\n'
        f'Description: {class_descriptions[class_name]}',
        fontsize=13, color='blue'
    )

    plt.tight_layout()
    plt.show()

    print(f"\n✅ Predicted Class : {class_name}")
    print(f"✅ Confidence      : {confidence:.2f}%")
    print(f"✅ Description     : {class_descriptions[class_name]}")
else:
    print("❌ No file uploaded.")